# Part 3: 데이터 분석 자동화 - RAG Chatbot

CSV → DuckDB(RDB), MD → ChromaDB(VectorDB)를 구축한 뒤,
LangChain / LangGraph / ReAct 3가지 방식으로 분석 챗봇을 구현하고 비교한다.

## 목차
### 3.1 환경 설정
### 3.2 데이터 준비
- 3.2.1 CSV → DuckDB (관계형 DB)
- 3.2.2 MD → ChromaDB (벡터 DB)
### 3.3 검색 함수 (공통 인터페이스)
- search_md(), search_csv(), route_question(), generate_final()
### 3.4 LangChain (LCEL) - 둘 다 실행 방식
### 3.5 LangGraph - 라우터 선택 방식
### 3.6 LangGraph ReAct - 재질문 방식
### 3.7 방식별 비교 분석
### 3.8 Streamlit 데모

---

## 하이브리드 RAG 개념

| 데이터 유형 | 저장소 | 검색 방식 | 적합한 질문 |
|------------|--------|----------|------------|
| MD 문서 | ChromaDB (벡터) | 의미 유사도 검색 | "왜?", "어떻게?", "전략은?" |
| CSV 데이터 | DuckDB (SQL) | Text-to-SQL | "얼마?", "Top 3?", "평균은?" |

```
[질문] → [Tool Calling 라우터] → MD 검색 / CSV 검색 / Clarify → [답변 생성]
```

## 3.1 환경 설정

In [96]:
import os
import glob
import time
import base64
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 벡터 DB (MD 문서용)
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# SQL DB (CSV 데이터용)
import duckdb

# LLM
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

# LangGraph
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal, Optional

# Mermaid 다이어그램 렌더링
from IPython.display import display, HTML

def display_mermaid(graph: str):
    """Mermaid 다이어그램을 노트북에 이미지로 렌더링"""
    encoded = base64.urlsafe_b64encode(graph.encode('utf-8')).decode('ascii')
    url = f'https://mermaid.ink/svg/{encoded}'
    display(HTML(
        f'<img src="{url}" style="background:white; padding:16px; '
        f'border-radius:8px; max-width:600px; border:1px solid #e0e0e0;" />'
    ))


In [97]:
# --- 경로 설정 ---
RESULTS_PATH = './results'
CHROMA_PATH = './chroma_db_part3'
DUCKDB_PATH = './database/sales_analysis.db'

# --- LLM 초기화 ---
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# --- 파일 확인 ---
md_files = glob.glob(f'{RESULTS_PATH}/**/*.md', recursive=True)
csv_files = glob.glob(f'{RESULTS_PATH}/**/*.csv', recursive=True)

print(f'MD 파일: {len(md_files)}개')
for f in sorted(md_files):
    print(f'  {f}')

print(f'\nCSV 파일: {len(csv_files)}개')
for f in sorted(csv_files):
    print(f'  {f}')

MD 파일: 4개
  ./results/part1/Part1_QA_지식베이스.md
  ./results/part1/Part1_분석결과_요약.md
  ./results/part2/Part2_QA_지식베이스.md
  ./results/part2/Part2_예측분석_요약.md

CSV 파일: 21개
  ./results/part1/category_sales.csv
  ./results/part1/city_sales.csv
  ./results/part1/cluster_holiday_promo.csv
  ./results/part1/cluster_sales.csv
  ./results/part1/daily_sales.csv
  ./results/part1/growth_decomposition.csv
  ./results/part1/holiday_pattern_d3.csv
  ./results/part1/holiday_promo_comparison.csv
  ./results/part1/holiday_pure_effect.csv
  ./results/part1/monthly_sales.csv
  ./results/part1/promo_by_category.csv
  ./results/part1/promo_effect.csv
  ./results/part1/store_basket_analysis.csv
  ./results/part1/type_sales.csv
  ./results/part1/weekday_pattern.csv
  ./results/part1/yearly_sales.csv
  ./results/part1/yoy_growth.csv
  ./results/part2/feature_importance.csv
  ./results/part2/model_performance.csv
  ./results/part2/promo_scenario.csv
  ./results/part2/weekday_effect.csv


## 3.2 데이터 준비

### 3.2.1 CSV → DuckDB (관계형 DB)

CSV 파일을 DuckDB에 테이블로 적재한다. 테이블명은 `part1__`, `part2__` 등 prefix로 구분하여 충돌을 방지한다.

| prefix | 출처 | 예시 테이블 |
|--------|------|-----------|
| `part1__` | Part1 분석 결과 | `part1__yearly_performance`, `part1__category_performance` |
| `part2__` | Part2 롤링/계절성 결과 | `part2__stationarity_test`, `part2__rolling_summary` |

In [98]:
conn = duckdb.connect(DUCKDB_PATH)

csv_files = glob.glob(f'{RESULTS_PATH}/**/*.csv', recursive=True)

print('DuckDB 테이블 적재:')
for csv_path in sorted(csv_files):
    rel_path = csv_path.replace(RESULTS_PATH + '/', '')
    parts = rel_path.split('/')

    if len(parts) >= 2:
        prefix = parts[0]
        filename = os.path.splitext(parts[-1])[0]
        table_name = f'{prefix}__{filename}'
    else:
        filename = os.path.splitext(parts[0])[0]
        table_name = filename

    df = pd.read_csv(csv_path)
    conn.execute(f'CREATE OR REPLACE TABLE "{table_name}" AS SELECT * FROM df')
    print(f'  [+] {table_name} ({len(df)} rows, {len(df.columns)} cols)')

tables = conn.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='main'"
).fetchall()
print(f'\n총 {len(tables)}개 테이블 적재 완료')

DuckDB 테이블 적재:
  [+] part1__category_sales (33 rows, 3 cols)
  [+] part1__city_sales (22 rows, 3 cols)
  [+] part1__cluster_holiday_promo (17 rows, 11 cols)
  [+] part1__cluster_sales (17 rows, 4 cols)
  [+] part1__daily_sales (1669 rows, 7 cols)
  [+] part1__growth_decomposition (4 rows, 7 cols)
  [+] part1__holiday_pattern_d3 (7 rows, 3 cols)
  [+] part1__holiday_promo_comparison (2 rows, 3 cols)
  [+] part1__holiday_pure_effect (2 rows, 2 cols)
  [+] part1__monthly_sales (55 rows, 3 cols)
  [+] part1__promo_by_category (33 rows, 4 cols)
  [+] part1__promo_effect (2 rows, 3 cols)
  [+] part1__store_basket_analysis (54 rows, 7 cols)
  [+] part1__type_sales (5 rows, 4 cols)
  [+] part1__weekday_pattern (7 rows, 4 cols)
  [+] part1__yearly_sales (5 rows, 2 cols)
  [+] part1__yoy_growth (12 rows, 5 cols)
  [+] part2__feature_importance (28 rows, 2 cols)
  [+] part2__model_performance (4 rows, 5 cols)
  [+] part2__promo_scenario (5 rows, 4 cols)
  [+] part2__weekday_effect (7 rows, 3 cols

In [99]:
def get_table_schemas() -> str:
    """DuckDB 전체 테이블 스키마를 문자열로 반환"""
    tables = conn.execute(
        "SELECT table_name FROM information_schema.tables WHERE table_schema='main'"
    ).fetchall()

    schema_parts = []
    for (table_name,) in tables:
        cols = conn.execute(
            f"SELECT column_name, data_type FROM information_schema.columns "
            f"WHERE table_name='{table_name}'"
        ).fetchall()
        col_str = ', '.join([f'{c[0]}({c[1]})' for c in cols])

        try:
            sample = conn.execute(f'SELECT * FROM "{table_name}" LIMIT 2').fetchdf()
            sample_str = sample.to_string(index=False)
        except:
            sample_str = '(조회 실패)'

        schema_parts.append(
            f'테이블: {table_name}\n'
            f'  컬럼: {col_str}\n'
            f'  예시:\n{sample_str}'
        )

    return '\n\n'.join(schema_parts)

print(get_table_schemas()[:6000])

테이블: part1__category_sales
  컬럼: family(VARCHAR), total_sales(DOUBLE), avg_sales(DOUBLE)
  예시:
   family  total_sales   avg_sales
GROCERY I 3.397525e+08 3769.750550
BEVERAGES 2.141446e+08 2376.057775

테이블: part1__city_sales
  컬럼: city(VARCHAR), avg_daily_sales(DOUBLE), total_sales(DOUBLE)
  예시:
    city  avg_daily_sales  total_sales
  Ambato     11949.120612 3.988616e+07
Babahoyo     10500.177339 1.752480e+07

테이블: part1__cluster_holiday_promo
  컬럼: cluster(BIGINT), A_비휴일_프로모X(DOUBLE), A_n(BIGINT), B_비휴일_프로모O(DOUBLE), B_n(BIGINT), C_휴일_프로모X(DOUBLE), C_n(BIGINT), D_휴일_프로모O(DOUBLE), D_n(BIGINT), promo_effect(DOUBLE), holiday_effect(DOUBLE)
  예시:
 cluster  A_비휴일_프로모X    A_n  B_비휴일_프로모O   B_n  C_휴일_프로모X   C_n   D_휴일_프로모O  D_n  promo_effect  holiday_effect
       1  154.171222 119275  947.028577 31535 154.471722 10898 1084.850051 3523    792.857355        0.300499
       2   99.367701  81430  932.780923 19022  90.941122  7499  930.120274 2203    833.413222       -8.426579

테이블: part1__clust

### 3.2.2 MD → ChromaDB (벡터 DB)

MD 문서를 청킹 → 임베딩 → ChromaDB에 저장한다.

- **임베딩 모델**: `intfloat/multilingual-e5-small` (한국어 지원, 경량)
- **청킹**: H2/H3 기준 분할 (chunk_size=800, overlap=100)
- **persist**: `persist_directory` 고정으로 재실행 시 재인덱싱 방지

In [100]:
embedding_model = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-small')

if os.path.exists(CHROMA_PATH) and os.listdir(CHROMA_PATH):
    vectorstore = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedding_model
    )
    print(f'기존 ChromaDB 로드: {vectorstore._collection.count()}개 청크')
else:
    md_files = glob.glob(f'{RESULTS_PATH}/**/*.md', recursive=True)

    docs = []
    for md_path in md_files:
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        filename = os.path.basename(md_path)
        docs.append(Document(page_content=content, metadata={'source': filename}))
        print(f'  로딩: {filename} ({len(content)} chars)')

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=['\n## ', '\n### ', '\n\n', '\n']
    )
    chunks = splitter.split_documents(docs)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=CHROMA_PATH
    )
    print(f'\nChromaDB 생성 완료: {len(chunks)}개 청크 저장 → {CHROMA_PATH}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


기존 ChromaDB 로드: 13개 청크


## 3.3 검색 함수 (공통 인터페이스)

세 가지 방식(3.4/3.5/3.6)이 **동일한 검색 함수**를 재사용한다.
오케스트레이션(어떤 순서로, 어떤 조합으로 호출할지)만 다르다.

### 통일된 출력 스키마

| 함수 | 반환 형태 | 용도 |
|------|----------|------|
| `search_md()` | `{type:"md", docs:[...], context:str}` | 의미 기반 문서 검색 |
| `search_csv()` | `{type:"csv", sql:str, rows:list, summary:str}` | Text-to-SQL 데이터 조회 |
| `route_question()` | `{route:"csv"\|"md"\|"clarify", confidence, reason}` | Tool Calling 라우팅 |
| `generate_final()` | `{answer:str, citations:list, debug:dict}` | 최종 답변 생성 |

### Tool Calling이란?

LLM에게 "어떤 도구를 쓸 수 있는지" 알려주면, LLM이 **질문에 맞는 도구를 직접 선택**합니다.

| | 텍스트 파싱 방식 | Tool Calling 방식 |
|---|---|---|
| LLM 응답 | `"query_sql"` (텍스트) | `tool_calls: [{name, args}]` (구조화) |
| 파싱 | 직접 문자열 비교 필요 | API가 자동 파싱 |
| 오류 가능성 | LLM이 예상 외 텍스트 출력 시 실패 | 스키마 강제로 안정적 |

3.5/3.6에서 `llm.bind_tools(tools)`로 도구를 등록하여 사용합니다.

In [101]:
def search_md(question: str, k: int = 3) -> dict:
    """ChromaDB 의미 유사도 검색 → 통일 스키마 반환"""
    results = vectorstore.similarity_search_with_score(question, k=k)

    docs = []
    for doc, score in results:
        docs.append({
            'source': doc.metadata.get('source', ''),
            'score': round(float(score), 4),
            'snippet': doc.page_content[:200]
        })

    context = '\n---\n'.join([doc.page_content for doc, _ in results])

    return {
        'type': 'md',
        'docs': docs,
        'context': context
    }

# 테스트
result = search_md('프로모션 전략은?')
print(f'검색 결과: {len(result["docs"])}개 문서')
for d in result['docs']:
    print(f'  [{d["score"]:.4f}] {d["source"]}: {d["snippet"][:80]}...')

검색 결과: 3개 문서
  [0.2741] Part1_분석결과_요약.md: ## 6. 전략적 권고사항

### 6.1 우선순위별 전략
| 우선순위 | 전략 | 근거 | 예상 효과 |
|---------|------|--...
  [0.2882] Part1_QA_지식베이스.md: ## Q4. 어떤 클러스터에 마케팅 예산을 집중해야 하나요?

**A:** 프로모션 효과가 높은 클러스터에 집중해야 합니다.
- **C5**: ...
  [0.2896] Part2_QA_지식베이스.md: # Part 2: 예측 모델 Q&A 지식베이스

## 모델 관련

### Q: 가장 성능이 좋은 모델은?
A: CatBoost (Test MAP...


In [102]:
def search_csv(question: str) -> dict:
    """LLM이 SQL 생성 → DuckDB 실행 → 통일 스키마 반환"""
    schema = get_table_schemas()

    sql_prompt = ChatPromptTemplate.from_messages([
        ('system',
         '당신은 DuckDB SQL 전문가입니다. 스키마를 참고하여 질문에 답하는 SQL을 작성하세요.\n\n'
         '{schema}\n\n'
         '규칙:\n'
         '- SELECT 문만 작성\n'
         '- SQL만 출력 (설명 없이)\n'
         '- 테이블명은 큰따옴표로 감싸기'),
        ('human', '{question}')
    ])

    try:
        response = llm.invoke(
            sql_prompt.format_messages(schema=schema, question=question)
        )
        sql = response.content.strip()
        sql = sql.replace('```sql', '').replace('```', '').strip()

        result_df = conn.execute(sql).fetchdf()
        rows = result_df.head(20).to_dict('records')
        summary = result_df.to_string(index=False)

        return {
            'type': 'csv',
            'sql': sql,
            'rows': rows,
            'summary': summary
        }

    except Exception as e:
        return {
            'type': 'csv',
            'sql': '',
            'rows': [],
            'summary': f'SQL 실행 실패: {str(e)}'
        }

# 테스트
result = search_csv('연도별 매출 합계는?')
print(f'SQL: {result["sql"]}')
print(f'결과: {len(result["rows"])}행')
if result['rows']:
    print(result['summary'][:500])


# --- Tool Calling용 래퍼 ---
from langchain_core.tools import tool

@tool
def tool_search_md(question: str) -> str:
    """분석 설명, 전략, 인사이트 등 정성적 답변이 필요할 때 MD 문서를 검색합니다."""
    result = search_md(question)
    return result['context']

@tool
def tool_search_csv(question: str) -> str:
    """숫자, 통계, 비교, 순위, 합계 등 정량적 데이터가 필요할 때 SQL로 검색합니다."""
    result = search_csv(question)
    return result['summary']

tools = [tool_search_md, tool_search_csv]
llm_with_tools = llm.bind_tools(tools)

print('Tool Calling 도구 등록 완료')
print(f'  등록된 도구: {[t.name for t in tools]}')
print(f'  LLM이 질문에 따라 도구를 자동 선택합니다.')


SQL: SELECT year, total_sales FROM "part1__yearly_sales";
결과: 5행
 year  total_sales
 2013 1.404190e+08
 2014 2.094742e+08
 2015 2.408801e+08
 2016 2.886545e+08
 2017 1.817837e+08
Tool Calling 도구 등록 완료
  등록된 도구: ['tool_search_md', 'tool_search_csv']
  LLM이 질문에 따라 도구를 자동 선택합니다.


### Tool Calling 라우팅

`llm.bind_tools()`로 도구를 등록하면, LLM이 질문에 맞는 **도구를 직접 선택**합니다.

```
질문 → LLM(bind_tools) → tool_calls 반환 → 경로 확정
```

- LLM이 `tool_search_csv` 또는 `tool_search_md`를 **구조화된 형태**로 선택
- 텍스트 파싱이 필요 없어 오류 가능성이 낮음
- `reason`에 tool_call 정보를 담아 **왜 그 도구를 선택했는지** 추적 가능

In [103]:
def route_question(question: str) -> dict:
    """Tool Calling 라우팅: LLM이 도구를 직접 선택"""
    response = llm_with_tools.invoke(question)

    if response.tool_calls:
        tc = response.tool_calls[0]
        route = 'csv' if 'csv' in tc['name'] else 'md'
        return {
            'route': route,
            'confidence': 0.9,
            'reason': f'Tool Call: {tc["name"]}({tc["args"]})'
        }

    return {'route': 'md', 'confidence': 0.5, 'reason': 'Tool Call 없음 → 기본 md'}

# 테스트
test_questions = [
    '연도별 매출 합계는 얼마야?',
    '프로모션 전략을 왜 그렇게 추천해?',
    '매출 분석해줘',
]
for q in test_questions:
    r = route_question(q)
    print(f'  [{r["route"]:7s}] (conf={r["confidence"]}) {q}')
    print(f'           → {r["reason"]}')


  [csv    ] (conf=0.9) 연도별 매출 합계는 얼마야?
           → Tool Call: tool_search_csv({'question': '연도별 매출 합계'})
  [md     ] (conf=0.9) 프로모션 전략을 왜 그렇게 추천해?
           → Tool Call: tool_search_md({'question': '프로모션 전략 추천 이유'})
  [csv    ] (conf=0.9) 매출 분석해줘
           → Tool Call: tool_search_csv({'question': '매출 분석'})


In [104]:
def generate_final(question: str, search_results: list) -> dict:
    """검색 결과를 종합하여 최종 답변 생성"""
    context_parts = []
    citations = []

    for result in search_results:
        if result['type'] == 'md':
            context_parts.append(f'[문서 검색 결과]\n{result["context"]}')
            citations.extend([d['source'] for d in result.get('docs', [])])
        elif result['type'] == 'csv':
            if result['rows']:
                context_parts.append(
                    f'[데이터 조회 결과]\nSQL: {result["sql"]}\n결과:\n{result["summary"]}'
                )
            else:
                context_parts.append(f'[데이터 조회 실패] {result["summary"]}')

    context = '\n\n'.join(context_parts) if context_parts else '검색 결과 없음'

    answer_prompt = ChatPromptTemplate.from_messages([
        ('system',
         '당신은 매출 데이터 분석 전문가입니다. 검색 결과를 바탕으로 질문에 답변하세요.\n\n'
         '규칙:\n'
         '- 검색 결과에 있는 정보만 사용\n'
         '- 숫자는 정확히 인용\n'
         '- 한국어로 답변\n\n'
         '{context}'),
        ('human', '{question}')
    ])

    answer = llm.invoke(
        answer_prompt.format_messages(context=context, question=question)
    ).content

    return {
        'answer': answer,
        'citations': list(set(citations)),
        'debug': {
            'context_length': len(context),
            'sources_used': [r['type'] for r in search_results]
        }
    }

## 3.4 LangChain (LCEL) - 둘 다 실행 방식

**컨셉**: 라우팅 없이 MD + CSV **둘 다** 실행한 뒤, 결과를 합쳐서 답변을 생성한다.

"순서"가 아니라 "전수 실행 후 통합"이 핵심이다.
### LangChain LCEL의 특징

| 항목 | 설명 |
|------|------|
| 라우팅 | 없음 (항상 둘 다 실행) |
| 장점 | 구현 단순, 정보 누락 없음 |
| 단점 | 불필요한 검색 비용, 속도 느림 |
| 적합한 상황 | "어떤 검색이 필요할지 모를 때" |


In [105]:
display_mermaid("""
flowchart LR
    Q["질문"] --> RunBoth["둘 다 실행"]
    RunBoth --> RunMD["search_md()"]
    RunBoth --> RunCSV["search_csv()"]
    RunMD --> Merge["Merge<br/>(결과 통합)"]
    RunCSV --> Merge
    Merge --> Generate["generate_final()"]
    Generate --> A["답변"]
""")


In [106]:
# --- LangChain LCEL 체인 구성 ---

def both_search(question: str) -> dict:
    """MD + CSV 둘 다 검색"""
    md_result = search_md(question)
    csv_result = search_csv(question)
    return {'question': question, 'search_results': [md_result, csv_result]}

def merge_and_generate(inputs: dict) -> dict:
    """검색 결과 통합 → 답변 생성"""
    return generate_final(inputs['question'], inputs['search_results'])

# LCEL 체인: search → merge → generate
langchain_chain = RunnableLambda(both_search) | RunnableLambda(merge_and_generate)

def ask_langchain(question: str) -> dict:
    """LangChain LCEL 방식으로 질문에 답변"""
    start = time.time()
    result = langchain_chain.invoke(question)
    elapsed = time.time() - start

    return {
        'method': 'LangChain (LCEL)',
        'answer': result['answer'],
        'citations': result['citations'],
        'elapsed': round(elapsed, 2),
        'debug': result['debug']
    }

print('LangChain LCEL 체인 구성 완료')
print(f'  체인 구조: both_search → merge_and_generate')

LangChain LCEL 체인 구성 완료
  체인 구조: both_search → merge_and_generate


In [107]:
# --- LangChain 테스트 ---
test_questions = [
    '연도별 매출 합계는?',
    '프로모션 전략을 추천해줘',
    '가장 성능이 좋은 모델은?',
]

for q in test_questions:
    print(f'\n질문: {q}')
    print('-' * 60)
    result = ask_langchain(q)
    print(f'답변: {result["answer"][:300]}...')
    print(f'소요 시간: {result["elapsed"]}초')
    print(f'사용 소스: {result["debug"]["sources_used"]}')


질문: 연도별 매출 합계는?
------------------------------------------------------------
답변: 연도별 매출 합계는 다음과 같습니다:

- 2013년: 140,419,000
- 2014년: 209,474,200
- 2015년: 240,880,100
- 2016년: 288,654,500
- 2017년: 181,783,700...
소요 시간: 4.87초
사용 소스: ['md', 'csv']

질문: 프로모션 전략을 추천해줘
------------------------------------------------------------
답변: 프로모션 전략에 대한 추천은 다음과 같습니다:

1. **프로모션 +50% 수준 유지**: 한계효과를 체감할 수 있는 수준으로 프로모션을 유지합니다.
2. **평일(월~목) 프로모션 집중**: 주말(금~일)보다 약 2.7배 효과적인 평일에 프로모션을 집중합니다.
3. **목요일 프로모션 강화**: 목요일에 +0.7%의 효과가 있으므로, 이 날의 프로모션을 강화합니다.
4. **수요일 프로모션 재검토**: 수요일은 -0.4%의 역효과가 있으므로, 이 날의 프로모션 전략을 재검토합니다.
5. **대형 매장 프로모션 유지**: 대형 매장...
소요 시간: 6.41초
사용 소스: ['md', 'csv']

질문: 가장 성능이 좋은 모델은?
------------------------------------------------------------
답변: 가장 성능이 좋은 모델은 CatBoost로, Test MAPE는 약 6%이며 R²는 0.76입니다....
소요 시간: 2.83초
사용 소스: ['md', 'csv']


## 3.5 LangGraph - 라우터 선택 방식

**컨셉**: `llm.bind_tools()`로 도구를 등록하면, LLM이 질문에 맞는 도구를 **직접 선택**하여 MD **또는** CSV **중 하나만** 실행한다.
### LangGraph Router의 특징

| 항목 | 설명 |
|------|------|
| 라우팅 | Tool Calling (LLM이 도구 직접 선택) |
| 장점 | 불필요한 검색 절약, 구조화된 도구 선택으로 안정적 |
| 단점 | 라우팅 오류 시 정보 누락 |
| 적합한 상황 | "질문 유형이 분명할 때" |

### LangChain과의 차이점

| | LangChain (3.4) | LangGraph (3.5) |
|--|-----------------|------------------|
| 도구 선택 | 없음 (둘 다 실행) | Tool Calling 1회 |
| 비용 | 높음 (2배 검색) | 낮음 |
| 상태 관리 | 없음 | StateGraph |
| 분기/흐름 | 고정 파이프라인 | 조건부 엣지 |


In [108]:
display_mermaid("""
flowchart TD
    Q["질문"] --> LLM["LLM<br/>(bind_tools)"]
    LLM -->|"tool_calls"| Route["도구 선택<br/>(tool_search_md / tool_search_csv)"]
    Route -->|"md"| SearchMD["search_md()"]
    Route -->|"csv"| SearchCSV["search_csv()"]
    SearchMD --> Generate["generate_final()"]
    SearchCSV --> Generate
    Generate --> A["답변"]
""")


In [109]:
# --- LangGraph: State 정의 ---

class RouterState(TypedDict):
    question: str
    route: str
    confidence: float
    reason: str
    search_result: dict
    answer: str
    citations: list
    debug: dict

# --- 노드 함수들 ---

def router_node(state: RouterState) -> dict:
    """Tool Calling 라우터 (bind_tools)"""
    routing = route_question(state['question'])
    return {
        'route': routing['route'],
        'confidence': routing['confidence'],
        'reason': routing['reason']
    }

def search_md_node(state: RouterState) -> dict:
    """MD 검색 실행"""
    return {'search_result': search_md(state['question'])}

def search_csv_node(state: RouterState) -> dict:
    """CSV 검색 실행"""
    return {'search_result': search_csv(state['question'])}

def generate_node(state: RouterState) -> dict:
    """통일된 스키마로 답변 생성 (어떤 검색이든 동일 로직)"""
    result = generate_final(state['question'], [state['search_result']])
    return {
        'answer': result['answer'],
        'citations': result['citations'],
        'debug': result['debug']
    }

def route_decision(state: RouterState) -> str:
    """라우팅 결과에 따른 분기"""
    if state['route'] == 'csv':
        return 'csv'
    return 'md'

# --- 그래프 구성 ---

graph_builder = StateGraph(RouterState)

graph_builder.add_node('router', router_node)
graph_builder.add_node('search_md', search_md_node)
graph_builder.add_node('search_csv', search_csv_node)
graph_builder.add_node('generate', generate_node)

graph_builder.set_entry_point('router')
graph_builder.add_conditional_edges(
    'router',
    route_decision,
    {'md': 'search_md', 'csv': 'search_csv'}
)
graph_builder.add_edge('search_md', 'generate')
graph_builder.add_edge('search_csv', 'generate')
graph_builder.add_edge('generate', END)

langgraph_app = graph_builder.compile()

def ask_langgraph(question: str) -> dict:
    """LangGraph 라우터 방식으로 질문에 답변"""
    start = time.time()
    result = langgraph_app.invoke({
        'question': question,
        'route': '', 'confidence': 0.0, 'reason': '',
        'search_result': {}, 'answer': '', 'citations': [], 'debug': {}
    })
    elapsed = time.time() - start

    return {
        'method': 'LangGraph (Router)',
        'answer': result['answer'],
        'route': result['route'],
        'reason': result['reason'],
        'citations': result['citations'],
        'elapsed': round(elapsed, 2),
        'debug': result['debug']
    }

print('LangGraph Router 그래프 구성 완료')
print(f'  노드: router → search_md/search_csv → generate → END')

LangGraph Router 그래프 구성 완료
  노드: router → search_md/search_csv → generate → END


In [110]:
# --- LangGraph 테스트 ---
test_questions = [
    '연도별 매출 합계는?',
    '프로모션 전략을 추천해줘',
    '가장 성능이 좋은 모델은?',
]

for q in test_questions:
    print(f'\n질문: {q}')
    print('-' * 60)
    result = ask_langgraph(q)
    print(f'라우팅: {result["route"]} ({result["reason"]})')
    print(f'답변: {result["answer"][:300]}...')
    print(f'소요 시간: {result["elapsed"]}초')


질문: 연도별 매출 합계는?
------------------------------------------------------------
라우팅: csv (Tool Call: tool_search_csv({'question': '연도별 매출 합계'}))
답변: 연도별 매출 합계는 다음과 같습니다:

- 2013년: 140,419,000
- 2014년: 209,474,200
- 2015년: 240,880,100
- 2016년: 288,654,500
- 2017년: 181,783,700...
소요 시간: 3.88초

질문: 프로모션 전략을 추천해줘
------------------------------------------------------------
라우팅: md (Tool Call: tool_search_md({'question': '프로모션 전략 추천'}))
답변: 프로모션 전략에 대한 추천은 다음과 같습니다:

1. **프로모션 +50% 수준 유지**: 한계효과를 체감할 수 있는 수준으로 프로모션을 유지합니다.

2. **평일(월~목) 프로모션 집중**: 주말(금~일)보다 약 2.7배 효과적인 평일에 프로모션을 집중적으로 진행합니다.

3. **목요일 프로모션 강화**: 목요일에 프로모션을 강화하여 +0.7%의 효과를 극대화합니다.

4. **수요일 프로모션 재검토**: 수요일은 -0.4%의 역효과가 있으므로, 프로모션 전략을 재검토해야 합니다.

5. **대형 매장 프로모션 유지**: 대형...
소요 시간: 5.99초

질문: 가장 성능이 좋은 모델은?
------------------------------------------------------------
라우팅: md (Tool Call: tool_search_md({'question': '가장 성능이 좋은 모델은?'}))
답변: 가장 성능이 좋은 모델은 CatBoost로, Test MAPE는 약 6%이며 R²는 0.76입니다....
소요 시간: 2.31초


## 3.6 LangGraph ReAct - 재질문 방식

**컨셉**: LLM이 **Tool Calling으로 도구를 선택**하고, 질문이 불명확하면 **Clarify → 질문 재작성 → 재실행**한다.

### ReAct를 3단계로 이해하기

ReAct(Reasoning + Acting)는 **사람이 문제를 푸는 과정**과 같습니다. 한 번에 전체를 이해하기보다, 3단계로 나눠서 살펴봅시다.

**Step A. 핵심 아이디어 — "생각하고, 행동하고, 관찰한다"**

사람이 질문에 답하는 과정을 떠올려 보세요:
1. "이 질문에 답하려면 뭐가 필요하지?" → **Thought (생각)**
2. "DB에서 데이터를 꺼내자" → **Action (행동)**
3. "결과가 5행 나왔네, 충분하다" → **Observation (관찰)**

이 3단계를 **정보가 충분할 때까지 반복**하는 것이 ReAct의 전부입니다.

**Step B. 우리 챗봇에 적용하면? (Tool Calling)**

위 개념을 LLM 챗봇에 적용하면:
- **Plan**: LLM이 `bind_tools()`로 등록된 도구 중 하나를 **Tool Call로 선택**
- **Act**: 선택된 도구를 `tool_calls`에서 꺼내 실행 (SQL 검색 or 문서 검색)
- **Observe**: 결과를 확인하고, 부족하면 Plan으로 돌아감

3.5와의 차이: 3.5는 Tool Calling **1회**, 3.6은 Tool Calling **루프** (최대 N회)

**Step C. 전체 흐름 (clarify 포함)**

여기에 "질문이 모호할 때 되묻기(clarify)" 기능까지 추가하면 최종 구조가 됩니다.
LLM이 tool_call 없이 텍스트만 반환하면 → clarify로 처리합니다.

### ReAct의 특징

| 항목 | 설명 |
|------|------|
| 도구 선택 | Tool Calling으로 매 단계 도구 선택 |
| Clarify | tool_call 없이 텍스트 반환 시 → 사용자에게 재질문 |
| 질문 재작성 | 사용자 응답을 반영하여 질문을 명확하게 재구성 |
| 루프 | Plan → Act → Observe → Plan (최대 N회) |

### 3가지 방식 비교

| | LangChain (3.4) | LangGraph (3.5) | ReAct (3.6) |
|--|-----------------|-----------------|-------------|
| 도구 선택 | 없음 (둘 다 실행) | Tool Calling 1회 | Tool Calling 루프 |
| 재질문 | 불가 | 불가 | 가능 (clarify) |
| 재시도 | 불가 | 불가 | 가능 (루프) |
| 복잡도 | 낮음 | 중간 | 높음 |


In [111]:
# ── Step A: ReAct 핵심 루프 ──
# 사람이 문제를 푸는 과정: 생각 → 행동 → 관찰 → 반복

print('Step A. ReAct 핵심 아이디어')
print('  사람이 문제를 푸는 과정과 동일합니다.\n')

display_mermaid("""
flowchart LR
    Think["Thought<br/>(생각)"] --> Act["Action<br/>(행동)"]
    Act --> Observe["Observation<br/>(관찰)"]
    Observe -->|"반복"| Think
""")

# ── Step B: Tool Calling으로 구현한 ReAct ──

print('\nStep B. Tool Calling으로 구현한 ReAct')
print('  LLM이 bind_tools()로 등록된 도구를 tool_call로 선택합니다.\n')

display_mermaid("""
flowchart LR
    Plan["Plan<br/>(도구 선택)"] -->|"tool_call"| Act["Act<br/>(도구 실행)"]
    Act --> Plan
    Plan -->|"충분"| Answer["최종 답변"]
""")

# ── Step C: 전체 흐름 (clarify 포함) ──

print('\nStep C. 전체 흐름 (clarify 포함)')
print('  질문이 모호하면 재질문, 구체적이면 도구 실행, 충분하면 답변합니다.\n')

display_mermaid("""
flowchart TD
    Q["질문"] --> Plan["Plan<br/>(구체성 판단 → 도구 선택)"]
    Plan -->|"구체적"| Act["Act<br/>(도구 실행)"]
    Plan -->|"모호"| Clarify["재질문"]
    Act --> Plan
    Plan -->|"충분"| Answer["최종 답변"]
    Clarify -->|"사용자 응답"| Rewrite["질문 재작성"] --> Plan
""")


Step A. ReAct 핵심 아이디어
  사람이 문제를 푸는 과정과 동일합니다.




Step B. Tool Calling으로 구현한 ReAct
  LLM이 bind_tools()로 등록된 도구를 tool_call로 선택합니다.




Step C. 전체 흐름 (clarify 포함)
  질문이 모호하면 재질문, 구체적이면 도구 실행, 충분하면 답변합니다.



In [112]:
# --- ReAct: State 정의 ---

class ReActState(TypedDict):
    question: str
    original_question: str
    action: str
    tool_args: dict
    observation: str
    search_results: list
    answer: str
    citations: list
    clarify_question: str
    user_response: str
    iteration: int
    max_iterations: int
    debug_log: list

# --- 노드 함수들 ---

def plan_node(state: ReActState) -> dict:
    """LLM이 Tool Calling으로 다음 행동 결정 (2단계: 구체성 판단 → 도구 선택)"""
    log = list(state.get('debug_log', []))

    # --- 1단계: 질문 구체성 판단 (첫 iteration이고 이전 관찰이 없을 때만) ---
    if not state.get('observation') and state.get('iteration', 0) == 0:
        check_msg = [
            ('system',
             '사용자의 질문이 데이터 분석 도구를 호출할 만큼 구체적인지 판단하세요.\n'
             '구체적인 질문 예시: "연도별 매출 합계는?", "카테고리별 매출 비교해줘", "프로모션 전략을 추천해줘"\n'
             '모호한 질문 예시: "매출 분석해줘", "데이터 좀 봐줘", "분석해줘"\n\n'
             '구체적이면: SPECIFIC\n'
             '모호하면: VAGUE: (사용자에게 할 재질문)'),
            ('human', state['question'])
        ]
        check_response = llm.invoke(check_msg)
        check_text = check_response.content.strip()

        if check_text.startswith('VAGUE'):
            clarify_text = check_text.split(':', 1)[-1].strip() if ':' in check_text else '좀 더 구체적으로 질문해 주세요.'
            log.append(f'Plan → clarify (질문 모호: "{state["question"]}")')
            return {'action': 'clarify', 'tool_args': {}, 'clarify_question': clarify_text, 'debug_log': log}

    # --- 2단계: Tool Calling (기존 로직) ---
    messages = []

    system_msg = (
        '당신은 데이터 분석 도우미입니다. 질문에 답하기 위해 도구를 사용하세요.\n'
        '사용 가능한 데이터:\n'
        '- tool_search_csv: 숫자/통계/비교/순위 등 정량적 데이터\n'
        '- tool_search_md: 분석 설명/전략/인사이트 등 정성적 답변\n\n'
        '도구를 호출하세요. 충분한 정보가 모였다면 최종 답변을 텍스트로 작성하세요.'
    )

    if state.get('observation'):
        system_msg += f'\n\n이전 관찰: {state["observation"]}'

    messages.append(('system', system_msg))
    messages.append(('human', state['question']))

    response = llm_with_tools.invoke(messages)

    if response.tool_calls:
        tc = response.tool_calls[0]
        action = 'query_sql' if 'csv' in tc['name'] else 'search_md'
        tool_args = tc['args']
        log.append(f'Plan → {action} (Tool Call: {tc["name"]})')
        return {'action': action, 'tool_args': tool_args, 'debug_log': log}

    if state.get('search_results'):
        log.append('Plan → final (충분한 정보)')
        return {'action': 'final', 'tool_args': {}, 'debug_log': log}

    # tool_call 없고 검색 결과도 없으면 → clarify fallback
    clarify_text = response.content.strip() if response.content else '추가 정보가 필요합니다.'
    log.append(f'Plan → clarify (Tool Call 없음)')
    return {'action': 'clarify', 'tool_args': {}, 'clarify_question': clarify_text, 'debug_log': log}

def act_node(state: ReActState) -> dict:
    """도구 실행만 담당"""
    action = state['action']
    tool_args = state.get('tool_args', {})
    question = tool_args.get('question', state['question'])
    log = list(state.get('debug_log', []))
    results = list(state.get('search_results', []))

    if action == 'query_sql':
        result = search_csv(question)
        results.append(result)
        observation = f'SQL 결과: {result["summary"][:300]}'
        log.append(f'Act(query_sql) → {len(result["rows"])} rows')
    else:
        result = search_md(question)
        results.append(result)
        observation = f'문서 검색: {len(result["docs"])}개 문서'
        log.append(f'Act(search_md) → {len(result["docs"])} docs')

    return {
        'observation': observation,
        'search_results': results,
        'debug_log': log,
        'iteration': state.get('iteration', 0) + 1
    }

def rewrite_node(state: ReActState) -> dict:
    """사용자 응답을 반영하여 질문 재작성"""
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ('system',
         '원래 질문과 사용자의 추가 정보를 결합하여 명확한 질문으로 재작성하세요.\n'
         '재작성된 질문만 출력하세요.'),
        ('human', '원래 질문: {original}\n추가 정보: {response}')
    ])

    new_question = llm.invoke(
        rewrite_prompt.format_messages(
            original=state['original_question'],
            response=state['user_response']
        )
    ).content

    log = list(state.get('debug_log', [])) + [f'Rewrite → "{new_question}"']
    return {
        'question': new_question,
        'observation': '',
        'clarify_question': '',
        'debug_log': log
    }

def generate_react_node(state: ReActState) -> dict:
    """최종 답변 생성"""
    result = generate_final(state['question'], state.get('search_results', []))
    return {
        'answer': result['answer'],
        'citations': result['citations']
    }

def after_plan(state: ReActState) -> str:
    """Plan 결과에 따라 다음 노드 결정"""
    action = state['action']
    if action == 'clarify':
        return 'wait_input'
    if action == 'final':
        return 'generate'
    if state.get('iteration', 0) >= state.get('max_iterations', 3):
        return 'generate'
    return 'act'

# --- 그래프 구성 ---

react_builder = StateGraph(ReActState)

react_builder.add_node('plan', plan_node)
react_builder.add_node('act', act_node)
react_builder.add_node('rewrite', rewrite_node)
react_builder.add_node('generate', generate_react_node)

react_builder.set_entry_point('plan')
react_builder.add_conditional_edges(
    'plan',
    after_plan,
    {'act': 'act', 'generate': 'generate', 'wait_input': END}
)
react_builder.add_edge('act', 'plan')
react_builder.add_edge('rewrite', 'plan')
react_builder.add_edge('generate', END)

react_app = react_builder.compile()

def ask_react(question: str, user_response: str = None, prev_state: dict = None) -> dict:
    """ReAct 방식으로 질문에 답변 (clarify 시 user_response와 함께 재호출)"""
    start = time.time()

    if prev_state and user_response:
        prev_state['user_response'] = user_response
        rewritten = rewrite_node(prev_state)
        for k, v in rewritten.items():
            prev_state[k] = v
        result = react_app.invoke(prev_state)
    else:
        init_state = {
            'question': question,
            'original_question': question,
            'action': '',
            'tool_args': {},
            'observation': '',
            'search_results': [],
            'answer': '',
            'citations': [],
            'clarify_question': '',
            'user_response': '',
            'iteration': 0,
            'max_iterations': 3,
            'debug_log': []
        }
        result = react_app.invoke(init_state)

    elapsed = time.time() - start

    output = {
        'method': 'LangGraph (ReAct + Tool Calling)',
        'elapsed': round(elapsed, 2),
        'debug_log': result.get('debug_log', []),
    }

    if result.get('clarify_question'):
        output['needs_clarify'] = True
        output['clarify_question'] = result['clarify_question']
        output['_state'] = dict(result)
    else:
        output['answer'] = result.get('answer', '')
        output['citations'] = result.get('citations', [])

    return output

print('LangGraph ReAct (Tool Calling) 그래프 구성 완료')
print('  노드: plan(bind_tools) → act → (plan/generate/wait_input)')
print('  clarify → rewrite → plan (재질문 루프)')


LangGraph ReAct (Tool Calling) 그래프 구성 완료
  노드: plan(bind_tools) → act → (plan/generate/wait_input)
  clarify → rewrite → plan (재질문 루프)


In [113]:
# --- ReAct 테스트 1: 명확한 질문 ---
print('=== 명확한 질문 테스트 ===')
result = ask_react('연도별 매출 합계는?')

if result.get('needs_clarify'):
    print(f'Clarify 요청: {result["clarify_question"]}')
else:
    print(f'답변: {result["answer"][:300]}...')

print(f'소요 시간: {result["elapsed"]}초')
print(f'디버그 로그: {result["debug_log"]}')

=== 명확한 질문 테스트 ===
답변: 연도별 매출 합계는 다음과 같습니다:

- 2013년: 140,419,000
- 2014년: 209,474,200
- 2015년: 240,880,100
- 2016년: 288,654,500
- 2017년: 181,783,700...
소요 시간: 7.3초
디버그 로그: ['Plan → query_sql (Tool Call: tool_search_csv)', 'Act(query_sql) → 5 rows', 'Plan → final (충분한 정보)']


In [114]:
# --- ReAct 테스트 2: 모호한 질문 → Clarify → 재실행 ---
print('=== 모호한 질문 테스트 ===')
result = ask_react('매출 분석해줘')

if result.get('needs_clarify'):
    print(f'Clarify 요청: {result["clarify_question"]}')
    print()

    # 사용자 응답 시뮬레이션
    user_answer = '카테고리별 매출 비교가 궁금해'
    print(f'사용자 응답: {user_answer}')
    print()

    # 질문 재작성 → 재실행
    result2 = ask_react(
        question=None,
        user_response=user_answer,
        prev_state=result['_state']
    )
    print(f'답변: {result2.get("answer", "")[:300]}...')
    print(f'디버그 로그: {result2["debug_log"]}')
else:
    print(f'답변: {result["answer"][:300]}...')

print(f'총 소요 시간: {result["elapsed"]}초')

=== 모호한 질문 테스트 ===
Clarify 요청: 어떤 특정한 분석을 원하시나요? 예를 들어, "연도별 매출 합계는?" 또는 "카테고리별 매출 비교해줘"와 같은 구체적인 질문을 해주시면 더 도움이 될 것 같습니다.

사용자 응답: 카테고리별 매출 비교가 궁금해

답변: 네, 카테고리별 매출을 비교하여 분석해보겠습니다. 다음은 각 카테고리의 총 매출입니다:

1. **GROCERY I**: 3.397525e+08 (339,752,500)
2. **BEVERAGES**: 2.141446e+08 (214,144,600)
3. **PRODUCE**: 1.208491e+08 (120,849,100)
4. **CLEANING**: 9.654543e+07 (96,545,430)
5. **DAIRY**: 6.381317e+07 (63,813,170)
6. **BREAD/BAKERY**: 4.169753e+07 ...
디버그 로그: ['Plan → clarify (질문 모호: "매출 분석해줘")', 'Rewrite → "카테고리별 매출을 비교하여 분석해줄 수 있나요?"', 'Plan → query_sql (Tool Call: tool_search_csv)', 'Act(query_sql) → 20 rows', 'Plan → query_sql (Tool Call: tool_search_csv)', 'Act(query_sql) → 20 rows', 'Plan → query_sql (Tool Call: tool_search_csv)', 'Act(query_sql) → 20 rows', 'Plan → query_sql (Tool Call: tool_search_csv)']
총 소요 시간: 1.3초


## 3.7 방식별 비교 분석

동일한 질문 3개(정량/정성/모호)에 대해 3가지 방식을 실행하고 비교한다.

In [115]:
# --- 3가지 방식 비교 ---
comparison_questions = [
    ('정량', '연도별 매출 합계는?'),
    ('정성', '프로모션 전략을 추천해줘'),
    ('모호', '매출에 대해 알려줘'),
]

print('=' * 80)
print('방식별 비교 분석')
print('=' * 80)

comparison_results = []

for q_type, question in comparison_questions:
    print(f'\n{"=" * 80}')
    print(f'질문 [{q_type}]: {question}')
    print('=' * 80)

    # LangChain
    lc_result = ask_langchain(question)
    print(f'\n[LangChain] ({lc_result["elapsed"]}초)')
    print(f'  소스: {lc_result["debug"]["sources_used"]}')
    print(f'  답변: {lc_result["answer"][:200]}...')

    # LangGraph
    lg_result = ask_langgraph(question)
    print(f'\n[LangGraph] ({lg_result["elapsed"]}초) 라우팅: {lg_result["route"]}')
    print(f'  답변: {lg_result["answer"][:200]}...')

    # ReAct
    react_result = ask_react(question)
    if react_result.get('needs_clarify'):
        print(f'\n[ReAct] ({react_result["elapsed"]}초) → Clarify 요청')
        print(f'  재질문: {react_result["clarify_question"]}')
    else:
        print(f'\n[ReAct] ({react_result["elapsed"]}초)')
        print(f'  답변: {react_result.get("answer", "")[:200]}...')
    print(f'  로그: {react_result["debug_log"]}')

    comparison_results.append({
        'type': q_type,
        'question': question,
        'langchain_time': lc_result['elapsed'],
        'langgraph_time': lg_result['elapsed'],
        'langgraph_route': lg_result['route'],
        'react_time': react_result['elapsed'],
        'react_clarify': react_result.get('needs_clarify', False),
    })

# 요약 테이블
print('\n\n' + '=' * 80)
print('비교 요약')
print('=' * 80)
df_compare = pd.DataFrame(comparison_results)
print(df_compare.to_string(index=False))

방식별 비교 분석

질문 [정량]: 연도별 매출 합계는?

[LangChain] (5.04초)
  소스: ['md', 'csv']
  답변: 연도별 매출 합계는 다음과 같습니다:

- 2013년: 140,419,000
- 2014년: 209,474,200
- 2015년: 240,880,100
- 2016년: 288,654,500
- 2017년: 181,783,700

각 연도의 매출 합계는 위와 같습니다....

[LangGraph] (3.65초) 라우팅: csv
  답변: 연도별 매출 합계는 다음과 같습니다:

- 2013년: 140,419,000
- 2014년: 209,474,200
- 2015년: 240,880,100
- 2016년: 288,654,500
- 2017년: 181,783,700...

[ReAct] (6.02초)
  답변: 연도별 매출 합계는 다음과 같습니다:

- 2013년: 140,419,000
- 2014년: 209,474,200
- 2015년: 240,880,100
- 2016년: 288,654,500
- 2017년: 181,783,700...
  로그: ['Plan → query_sql (Tool Call: tool_search_csv)', 'Act(query_sql) → 5 rows', 'Plan → final (충분한 정보)']

질문 [정성]: 프로모션 전략을 추천해줘

[LangChain] (6.22초)
  소스: ['md', 'csv']
  답변: 프로모션 전략에 대한 추천은 다음과 같습니다:

1. **프로모션 +50% 수준 유지**: 한계효과를 체감할 수 있는 수준으로 프로모션을 유지합니다.
   
2. **평일(월~목) 프로모션 집중**: 평일이 주말보다 약 2.7배 효과적이므로, 월요일부터 목요일까지의 프로모션에 집중합니다.

3. **목요일 프로모션 강화**: 목요일은 +0.7%의 효과가 있...

[LangGraph] (5.71초) 라우팅: md
  답변: 프로모션 전략에 대한 추천은 다음과 같습니다:

1. *

### 비교 분석 정리

| 방식 | 핵심 차이 | 장점 | 단점 | 적합한 상황 |
|------|----------|------|------|----------|
| **LangChain (LCEL)** | 둘 다 실행 + Merge | 정보 누락 없음, 구현 단순 | 불필요한 검색 비용 | 질문 유형 예측 불가 |
| **LangGraph (Router)** | Tool Calling 1회로 도구 선택 | 비용 절약, 구조화된 선택 | 라우팅 오류 가능 | 질문 유형이 분명 |
| **LangGraph (ReAct)** | Tool Calling 루프 + Clarify | 모호한 질문 처리, 재시도 | 구현 복잡, 속도 느림 | 대화형 챗봇 |

**핵심 교훈**: 세 방식 모두 **동일한 검색 함수**(search_md, search_csv, generate_final)를 사용한다.
차이는 "오케스트레이션"(호출 순서, 조합, 분기 로직)뿐이다.

**진화 흐름**: 도구 선택 없음(3.4) → Tool Calling 1회(3.5) → Tool Calling 루프(3.6)


## 3.8 Streamlit 데모

3가지 방식을 선택하여 대화형으로 테스트할 수 있는 Streamlit 데모 앱을 생성한다.

### UI 구성
- **사이드바**: 방식 선택 (LangChain / LangGraph / ReAct)
- **메인**: 채팅 인터페이스
- **ReAct clarify**: "추가로 이것만 알려주세요" + text_input 1개 (단순하게)

In [116]:
%%writefile streamlit_app.py
import streamlit as st
import os
import glob
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import duckdb
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from typing import TypedDict

# --- 설정 ---
RESULTS_PATH = './results'
CHROMA_PATH = './chroma_db_part3'
DUCKDB_PATH = './data/sales_analysis_v2.db'

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

from langchain_core.tools import tool

@tool
def tool_search_md(question: str) -> str:
    """분석 설명, 전략, 인사이트 등 정성적 답변이 필요할 때 MD 문서를 검색합니다."""
    result = search_md(question)
    return result['context']

@tool
def tool_search_csv(question: str) -> str:
    """숫자, 통계, 비교, 순위, 합계 등 정량적 데이터가 필요할 때 SQL로 검색합니다."""
    result = search_csv(question)
    return result['summary']

tools = [tool_search_md, tool_search_csv]
llm_with_tools = llm.bind_tools(tools)

# --- DB 로드 ---
@st.cache_resource
def load_duckdb():
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    return conn

@st.cache_resource
def load_vectorstore():
    embedding_model = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-small')
    return Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_model)

conn = load_duckdb()
vectorstore = load_vectorstore()

# --- 검색 함수 (노트북과 동일) ---

def get_table_schemas():
    tables = conn.execute(
        "SELECT table_name FROM information_schema.tables WHERE table_schema='main'"
    ).fetchall()
    schema_parts = []
    for (table_name,) in tables:
        cols = conn.execute(
            f"SELECT column_name, data_type FROM information_schema.columns "
            f"WHERE table_name='{table_name}'"
        ).fetchall()
        col_str = ', '.join([f'{c[0]}({c[1]})' for c in cols])
        schema_parts.append(f'테이블: {table_name}\n  컬럼: {col_str}')
    return '\n\n'.join(schema_parts)

def search_md(question, k=3):
    results = vectorstore.similarity_search_with_score(question, k=k)
    docs = [{'source': doc.metadata.get('source', ''), 'score': round(float(score), 4),
             'snippet': doc.page_content[:200]} for doc, score in results]
    context = '\n---\n'.join([doc.page_content for doc, _ in results])
    return {'type': 'md', 'docs': docs, 'context': context}

def search_csv(question):
    schema = get_table_schemas()
    sql_prompt = ChatPromptTemplate.from_messages([
        ('system', '당신은 DuckDB SQL 전문가입니다. 스키마를 참고하여 SQL을 작성하세요.\n\n{schema}\n\n규칙: SELECT 문만, SQL만 출력, 테이블명은 큰따옴표로'),
        ('human', '{question}')
    ])
    try:
        sql = llm.invoke(sql_prompt.format_messages(schema=schema, question=question)).content.strip()
        sql = sql.replace('```sql', '').replace('```', '').strip()
        result_df = conn.execute(sql).fetchdf()
        return {'type': 'csv', 'sql': sql, 'rows': result_df.head(20).to_dict('records'), 'summary': result_df.to_string(index=False)}
    except Exception as e:
        return {'type': 'csv', 'sql': '', 'rows': [], 'summary': f'SQL 실행 실패: {str(e)}'}

def route_question(question):
    response = llm_with_tools.invoke(question)
    if response.tool_calls:
        tc = response.tool_calls[0]
        route = 'csv' if 'csv' in tc['name'] else 'md'
        return {'route': route, 'confidence': 0.9, 'reason': f'Tool Call: {tc["name"]}'}
    return {'route': 'md', 'confidence': 0.5, 'reason': 'Tool Call 없음 → 기본 md'}

def generate_final(question, search_results):
    context_parts = []
    citations = []
    for r in search_results:
        if r['type'] == 'md':
            context_parts.append(f'[문서 검색]\n{r["context"]}')
            citations.extend([d['source'] for d in r.get('docs', [])])
        elif r['type'] == 'csv' and r['rows']:
            context_parts.append(f'[데이터 조회]\nSQL: {r["sql"]}\n{r["summary"]}')
    context = '\n\n'.join(context_parts) if context_parts else '검색 결과 없음'
    answer_prompt = ChatPromptTemplate.from_messages([
        ('system', '매출 데이터 분석 전문가로서 검색 결과를 바탕으로 한국어로 답변하세요.\n\n{context}'),
        ('human', '{question}')
    ])
    answer = llm.invoke(answer_prompt.format_messages(context=context, question=question)).content
    return {'answer': answer, 'citations': list(set(citations))}

# --- 3가지 방식 ---

def ask_langchain(question):
    md_r = search_md(question)
    csv_r = search_csv(question)
    return generate_final(question, [md_r, csv_r])

def ask_langgraph(question):
    routing = route_question(question)
    if routing['route'] == 'csv':
        result = search_csv(question)
    else:
        result = search_md(question)
    out = generate_final(question, [result])
    out['route'] = routing['route']
    out['reason'] = routing['reason']
    return out

def ask_react(question):
    # 1단계: 질문 구체성 판단
    check_msg = [
        ('system',
         '사용자의 질문이 데이터 분석 도구를 호출할 만큼 구체적인지 판단하세요.\n'
         '구체적인 질문 예시: "연도별 매출 합계는?", "카테고리별 매출 비교해줘", "프로모션 전략을 추천해줘"\n'
         '모호한 질문 예시: "매출 분석해줘", "데이터 좀 봐줘", "분석해줘"\n\n'
         '구체적이면: SPECIFIC\n'
         '모호하면: VAGUE: (사용자에게 할 재질문)'),
        ('human', question)
    ]
    check = llm.invoke(check_msg).content.strip()
    if check.startswith('VAGUE'):
        clarify_text = check.split(':', 1)[-1].strip() if ':' in check else '좀 더 구체적으로 질문해 주세요.'
        return {'needs_clarify': True, 'clarify_question': clarify_text, 'route': 'clarify'}

    # 2단계: Tool Calling
    response = llm_with_tools.invoke(question)
    if response.tool_calls:
        tc = response.tool_calls[0]
        if 'csv' in tc['name']:
            result = search_csv(tc['args'].get('question', question))
        else:
            result = search_md(tc['args'].get('question', question))
        out = generate_final(question, [result])
        out['route'] = 'csv' if 'csv' in tc['name'] else 'md'
        out['needs_clarify'] = False
        return out
    return {'needs_clarify': True, 'clarify_question': response.content or '추가 정보가 필요합니다.', 'route': 'clarify'}

def ask_react_with_clarify(original_q, user_response):
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ('system', '원래 질문과 추가 정보를 결합하여 명확한 질문으로 재작성하세요. 질문만 출력.'),
        ('human', '원래 질문: {original}\n추가 정보: {response}')
    ])
    new_q = llm.invoke(rewrite_prompt.format_messages(original=original_q, response=user_response)).content
    return ask_react(new_q), new_q

# --- Streamlit UI ---

st.set_page_config(page_title='매출 분석 RAG Chatbot', page_icon='📊', layout='wide')
st.title('📊 매출 분석 RAG Chatbot')

method = st.sidebar.radio(
    '방식 선택',
    ['LangChain (둘 다 실행)', 'LangGraph (라우터 선택)', 'ReAct (재질문)'],
    index=1
)

st.sidebar.markdown('---')
st.sidebar.markdown('### 방식 설명')
if 'LangChain' in method:
    st.sidebar.info('MD + CSV 둘 다 검색 후 결과를 통합합니다.')
elif 'LangGraph' in method:
    st.sidebar.info('질문 유형을 판단하여 MD 또는 CSV 중 하나만 검색합니다.')
else:
    st.sidebar.info('질문이 모호하면 추가 질문 후 재실행합니다.')

if 'messages' not in st.session_state:
    st.session_state.messages = []
if 'waiting_clarify' not in st.session_state:
    st.session_state.waiting_clarify = False
if 'original_question' not in st.session_state:
    st.session_state.original_question = ''

for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])

if st.session_state.waiting_clarify:
    clarify_response = st.chat_input('추가 정보를 입력하세요...')
    if clarify_response:
        st.session_state.messages.append({'role': 'user', 'content': clarify_response})
        with st.chat_message('user'):
            st.markdown(clarify_response)

        with st.chat_message('assistant'):
            with st.spinner('질문 재작성 후 검색 중...'):
                result, new_q = ask_react_with_clarify(st.session_state.original_question, clarify_response)
                response = f'**[재작성된 질문]**: {new_q}\n\n{result["answer"]}'
                st.markdown(response)

        st.session_state.messages.append({'role': 'assistant', 'content': response})
        st.session_state.waiting_clarify = False
else:
    user_input = st.chat_input('매출 데이터에 대해 질문하세요...')
    if user_input:
        st.session_state.messages.append({'role': 'user', 'content': user_input})
        with st.chat_message('user'):
            st.markdown(user_input)

        with st.chat_message('assistant'):
            with st.spinner('검색 중...'):
                start = time.time()
                if 'LangChain' in method:
                    result = ask_langchain(user_input)
                elif 'LangGraph' in method:
                    result = ask_langgraph(user_input)
                else:
                    result = ask_react(user_input)
                elapsed = round(time.time() - start, 2)

            if result.get('needs_clarify'):
                response = f'🤔 **추가 정보가 필요합니다**\n\n{result["clarify_question"]}'
                st.markdown(response)
                st.session_state.waiting_clarify = True
                st.session_state.original_question = user_input
            else:
                route_info = f' (경로: {result.get("route", "all")})' if result.get('route') else ''
                response = f'{result["answer"]}\n\n---\n⏱ {elapsed}초{route_info}'
                st.markdown(response)

        st.session_state.messages.append({'role': 'assistant', 'content': response})

Overwriting streamlit_app.py


### Streamlit 실행 가이드

```bash
# 터미널에서 실행
cd store-sales-time-series-forecasting
streamlit run streamlit_app.py
```

### 주의사항
- 노트북에서 3.2 데이터 준비를 먼저 실행하여 DuckDB, ChromaDB를 생성해야 한다
- `.env`에 `OPENAI_API_KEY`가 설정되어 있어야 한다
- Streamlit 앱은 노트북과 동일한 검색 함수를 사용한다 (코드 재사용)

### ReAct Clarify 동작
1. 모호한 질문 입력 → "추가 정보가 필요합니다" 메시지 표시
2. 사용자가 추가 정보 입력
3. 질문 재작성 후 자동 재실행 → 최종 답변 표시